# 🔬 ADMET Toxicity Prediction: Milestone 3 — Explainability
## Understanding *why* ChemBERTa predicts toxicity

Two complementary attribution methods:

| Method | Basis | Speed |
|--------|-------|-------|
| **Attention** | CLS-token attention weights from the last transformer layer | Fast |
| **Integrated Gradients (IG)** | Formal Shapley-like attribution along the path from baseline → input | Slower, more principled |

Both methods produce per-token scores that are mapped back to RDKit atom indices for 2-D molecular visualization.

**Architecture reminder:**
```
SMILES → ChemBERTa-77M (RoBERTa backbone + LoRA, merged)
      → CLS embedding
      → Linear(600 → 12) → sigmoid → 12 toxicity probabilities
```

## Section 1: Setup

In [ ]:
!pip install -q captum

In [ ]:
import os, re, glob, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import SVG, HTML, display
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from captum.attr import LayerIntegratedGradients
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
BASE_MODEL = 'DeepChem/ChemBERTa-77M-MTR'

# Auto-detect the most recent multitarget checkpoint
checkpoints = sorted(glob.glob('chemberta-tox21-multitarget-*'))
MODEL_DIR = checkpoints[-1] if checkpoints else 'chemberta-tox21-multitarget'
print(f'Checkpoint: {MODEL_DIR}')

NUM_TARGETS = 12
TARGET_NAMES = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
]
TARGET_IDX = {name: i for i, name in enumerate(TARGET_NAMES)}

DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f'Device: {DEVICE}')

# Optimal thresholds from M2 training run
THRESHOLDS = {
    'NR-AR': 0.95, 'NR-AR-LBD': 0.95, 'NR-AhR': 0.75,
    'NR-Aromatase': 0.85, 'NR-ER': 0.70, 'NR-ER-LBD': 0.85,
    'NR-PPAR-gamma': 0.85, 'SR-ARE': 0.65, 'SR-ATAD5': 0.80,
    'SR-HSE': 0.85, 'SR-MMP': 0.85, 'SR-p53': 0.85,
}

## Section 2: Load the Trained Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# Load base model architecture, then apply the saved LoRA adapter
base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_TARGETS,
    ignore_mismatched_sizes=True,
)
model = PeftModel.from_pretrained(base, MODEL_DIR)
model = model.merge_and_unload()  # bake LoRA weights in; simplifies captum
model.eval()
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

# Locate embedding layer for Integrated Gradients
if hasattr(model, 'roberta'):
    EMBED_LAYER = model.roberta.embeddings
elif hasattr(model, 'bert'):
    EMBED_LAYER = model.bert.embeddings
else:
    EMBED_LAYER = model.base_model.embeddings
print(f'Embedding layer: {type(EMBED_LAYER).__name__}')

In [ ]:
def predict(smiles, verbose=True):
    enc = tokenizer(smiles, return_tensors='pt', truncation=True, max_length=512)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = torch.sigmoid(logits).cpu().numpy()
    if verbose:
        print(f'\nSMILES: {smiles}')
        print(f'{"Target":<16}{"Prob":>7}  Result')
        print('-' * 36)
        for name, prob in zip(TARGET_NAMES, probs):
            label = 'TOXIC' if prob >= THRESHOLDS[name] else 'safe'
            flag  = '⚠️ ' if label == 'TOXIC' else '   '
            print(f'{flag}{name:<14}{prob:>7.3f}  {label}')
    return dict(zip(TARGET_NAMES, probs.tolist()))

# Sanity check
print('=== Aspirin (should be mostly safe) ===')
_ = predict('CC(=O)Oc1ccccc1C(=O)O')

## Section 3: Attention Visualization

The transformer's CLS token aggregates information from every SMILES token via self-attention.
By examining the **last-layer attention weights** from CLS to each token (averaged over all heads),
we see which substructures the model 'looked at' most when producing its toxicity score.

> **Limitation:** attention weights show *where* the model looks, not *how much each position contributes*
> to the final prediction. Integrated Gradients (Section 4) provides a more principled answer.

In [ ]:
# ── SMILES atom-position parser ───────────────────────────────────────────
_ATOM_RE = re.compile(
    r'\[[^\]]+\]'   # bracketed atoms: [NH], [C@@H], [nH+], ...
    r'|Cl|Br'         # two-char elements
    r'|[BCNOPSFI]'    # uppercase single-char
    r'|[bcnops]'      # aromatic lowercase
)

def find_atom_positions(smiles):
    """Return (start, end) char positions for every atom in SMILES."""
    return [(m.start(), m.end()) for m in _ATOM_RE.finditer(smiles)]

def get_canonical(smiles):
    """Return (canonical_smiles, rdkit_mol)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Cannot parse: {smiles}')
    can = Chem.MolToSmiles(mol)
    return can, Chem.MolFromSmiles(can)

def get_encoding(canonical):
    """Tokenize canonical SMILES and return (input_ids, attn_mask, offsets)."""
    enc = tokenizer(
        canonical,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )
    input_ids = enc['input_ids'].to(DEVICE)
    attn_mask = enc['attention_mask'].to(DEVICE)
    offsets   = enc['offset_mapping'][0].tolist()
    return input_ids, attn_mask, offsets

def atom_to_token_map(canonical, offsets):
    """Map RDKit atom index -> token index."""
    positions = find_atom_positions(canonical)
    mapping = {}
    for atom_idx, (a_start, _) in enumerate(positions):
        for tok_idx, (t_start, t_end) in enumerate(offsets):
            if t_start <= a_start < t_end:
                mapping[atom_idx] = tok_idx
                break
    return mapping

In [ ]:
# ── Visualisation helpers ──────────────────────────────────────────────────
def scores_to_atom_dict(mol, atom_map, token_scores):
    """Pull per-token scores into a {atom_idx: score} dict."""
    return {
        i: float(token_scores[atom_map[i]])
        if i in atom_map and atom_map[i] < len(token_scores) else 0.0
        for i in range(mol.GetNumAtoms())
    }

def draw_2d(mol, atom_scores_dict, width=360, height=260):
    """Return SVG with atoms colour-coded by score (white → red via YlOrRd)."""
    n    = mol.GetNumAtoms()
    vals = np.array([atom_scores_dict.get(i, 0.0) for i in range(n)])
    pos  = np.clip(vals, 0, None)
    vmax = pos.max() if pos.max() > 0 else 1.0
    norm = pos / vmax
    cmap = plt.colormaps['YlOrRd']
    colors = {i: cmap(float(norm[i]))[:3] for i in range(n)}
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    drawer.drawOptions().addStereoAnnotation = False
    drawer.DrawMolecule(
        mol,
        highlightAtoms=list(range(n)),
        highlightAtomColors=colors,
        highlightBonds=[],
    )
    drawer.EndDrawing()
    return drawer.GetDrawingText()

def html_smiles(canonical, token_scores, offsets, title=''):
    """HTML with each SMILES character background-coloured by token score."""
    scores = np.array(token_scores)
    pos    = np.clip(scores, 0, None)
    vmax   = pos.max() if pos.max() > 0 else 1.0
    norm   = pos / vmax
    char_color = ['#ffffff'] * len(canonical)
    for tok_idx, (s, e) in enumerate(offsets):
        if tok_idx >= len(norm) or s == e:
            continue
        intensity = float(norm[tok_idx])
        r = 255; g = int(255 * (1 - intensity)); b = int(255 * (1 - intensity))
        hex_c = f'#{r:02x}{g:02x}{b:02x}'
        for ci in range(s, min(e, len(char_color))):
            char_color[ci] = hex_c
    spans = ''.join(
        f'<span style="background:{char_color[i]};padding:1px 0;'
        f'font-family:monospace;font-size:13px">{ch}</span>'
        for i, ch in enumerate(canonical)
    )
    return f'<div style="margin:10px 0"><b>{title}</b><br>{spans}</div>'

In [ ]:
def explain_attention(smiles, target_name='SR-p53'):
    """Extract CLS attention from final layer; display coloured SMILES + 2D."""
    canonical, mol = get_canonical(smiles)
    input_ids, attn_mask, offsets = get_encoding(canonical)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            output_attentions=True,
        )

    # Last layer, mean over heads, CLS row
    last_attn    = outputs.attentions[-1][0]   # [heads, seq, seq]
    cls_attn     = last_attn.mean(0)[0]        # [seq_len]
    token_scores = cls_attn.cpu().numpy()

    prob  = float(torch.sigmoid(outputs.logits[0, TARGET_IDX[target_name]]))
    label = 'TOXIC' if prob >= THRESHOLDS[target_name] else 'safe'

    a_map = atom_to_token_map(canonical, offsets)
    atom_scores = scores_to_atom_dict(mol, a_map, token_scores)

    header = f'{target_name}: {prob:.3f} ({label})  |  Attention'
    display(HTML(html_smiles(canonical, token_scores, offsets, title=f'SMILES — {header}')))
    display(SVG(draw_2d(mol, atom_scores)))
    return token_scores, atom_scores

In [ ]:
MOLECULES = {
    'Aspirin'      : 'CC(=O)Oc1ccccc1C(=O)O',
    'Tamoxifen'    : 'CCC(=C(c1ccccc1)c1ccc(OCCO)cc1)c1ccccc1',
    'Bisphenol A'  : 'CC(c1ccc(O)cc1)(c1ccc(O)cc1)',
    'Dioxin (TCDD)': 'Clc1ccc2c(c1)Oc1cc(Cl)ccc1O2',
    'Caffeine'     : 'Cn1c(=O)c2c(ncn2C)n(C)c1=O',
}

print('Attention — Tamoxifen → NR-ER')
_ = explain_attention(MOLECULES['Tamoxifen'], target_name='NR-ER')

print('\nAttention — Dioxin → SR-p53')
_ = explain_attention(MOLECULES['Dioxin (TCDD)'], target_name='SR-p53')

## Section 4: Integrated Gradients

**Integrated Gradients (Sundararajan et al., 2017)** computes the average gradient of the
prediction with respect to the *input embeddings* along a straight path from a neutral baseline
(all-padding) to the actual input.  Multiplying by `(input − baseline)` gives a formally
correct attribution that satisfies **completeness** (attributions sum to the prediction–baseline difference).

```
IG_i = (x_i - x'_i) × ∫_0^1 ∂F(x' + α(x-x')) / ∂x_i  dα
```

We integrate over the **embedding dimension** to get one scalar score per SMILES token,
then aggregate to atoms as before.

In [ ]:
def _forward_for_ig(input_ids, attention_mask, target_idx):
    """Scalar sigmoid probability for one target (captum forward function)."""
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    return torch.sigmoid(out.logits[:, target_idx])

lig = LayerIntegratedGradients(_forward_for_ig, EMBED_LAYER)

def explain_ig(smiles, target_name='SR-p53', n_steps=50):
    """Compute IG token attributions; display coloured SMILES + 2D structure."""
    canonical, mol = get_canonical(smiles)
    input_ids, attn_mask, offsets = get_encoding(canonical)
    target_idx   = TARGET_IDX[target_name]
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)

    attrs, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attn_mask, target_idx),
        n_steps=n_steps,
        return_convergence_delta=True,
    )
    # [1, seq_len, embed_dim] -> [seq_len]
    token_scores = attrs.sum(dim=-1).squeeze(0).detach().cpu().numpy()

    with torch.no_grad():
        prob = float(torch.sigmoid(
            model(input_ids=input_ids, attention_mask=attn_mask).logits[0, target_idx]
        ))
    label = 'TOXIC' if prob >= THRESHOLDS[target_name] else 'safe'

    a_map = atom_to_token_map(canonical, offsets)
    atom_scores = scores_to_atom_dict(mol, a_map, token_scores)

    header = f'{target_name}: {prob:.3f} ({label})  |  Integrated Gradients'
    display(HTML(html_smiles(canonical, token_scores, offsets, title=f'SMILES — {header}')))
    display(SVG(draw_2d(mol, atom_scores)))
    print(f'  convergence delta: {delta.mean().item():.5f}')
    return token_scores, atom_scores

In [ ]:
print('Integrated Gradients — Tamoxifen → NR-ER')
print('=' * 52)
_ = explain_ig(MOLECULES['Tamoxifen'], target_name='NR-ER')

print('\nIntegrated Gradients — Bisphenol A → NR-ER')
print('=' * 52)
_ = explain_ig(MOLECULES['Bisphenol A'], target_name='NR-ER')

print('\nIntegrated Gradients — Dioxin → SR-p53')
print('=' * 52)
_ = explain_ig(MOLECULES['Dioxin (TCDD)'], target_name='SR-p53')

## Section 5: Multi-Target Comparison

The same molecule can trigger different toxicity pathways via different substructures.
Here we run IG for **four targets** on the same compound and display the atom-level
attributions side-by-side.  Red = strongly positive contributor; white = neutral.

**What to look for:**
- Tamoxifen's triphenylethylene core drives NR-AR/NR-ER binding; the ethoxy tail is less relevant
- Bisphenol A's phenol groups should be highlighted for nuclear receptor targets
- Dioxin's chlorine substituents should dominate SR-p53 (DNA damage) attribution

In [ ]:
def compare_targets(smiles, mol_name, targets=('NR-ER', 'NR-AR', 'SR-p53', 'SR-MMP')):
    """Run IG for multiple targets; display as an HTML side-by-side grid."""
    canonical, mol = get_canonical(smiles)
    input_ids, attn_mask, offsets = get_encoding(canonical)
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    a_map = atom_to_token_map(canonical, offsets)

    cols = []
    print(f'\n{mol_name}')
    print(f'{"Target":<16}{"Prob":>7}  Result')
    print('-' * 32)
    for target_name in targets:
        target_idx = TARGET_IDX[target_name]
        attrs, _ = lig.attribute(
            inputs=input_ids,
            baselines=baseline_ids,
            additional_forward_args=(attn_mask, target_idx),
            n_steps=50,
            return_convergence_delta=True,
        )
        token_scores = attrs.sum(dim=-1).squeeze(0).detach().cpu().numpy()
        with torch.no_grad():
            prob = float(torch.sigmoid(
                model(input_ids=input_ids, attention_mask=attn_mask).logits[0, target_idx]
            ))
        label = 'TOXIC' if prob >= THRESHOLDS[target_name] else 'safe'
        flag  = '⚠️' if label == 'TOXIC' else '  '
        print(f'{flag} {target_name:<14}{prob:>7.3f}  {label}')
        atom_scores = scores_to_atom_dict(mol, a_map, token_scores)
        svg = draw_2d(mol, atom_scores)
        cols.append(f'<td style="text-align:center;padding:6px">'
                    f'<b>{target_name}</b><br>{prob:.3f} ({label})<br>{svg}</td>')
    display(HTML(f'<h4>{mol_name}</h4><table><tr>{chr(10).join(cols)}</tr></table>'))

compare_targets(MOLECULES['Tamoxifen'], 'Tamoxifen')
compare_targets(MOLECULES['Bisphenol A'], 'Bisphenol A')

## Section 6: Molecular Gallery

Run IG across all five test molecules for a single target.
Use `target_name` to switch between assays.

In [ ]:
def gallery(molecules_dict, target_name='SR-p53', n_steps=30):
    """IG gallery across all molecules for a single target."""
    print(f'Gallery — Target: {target_name}')
    print('=' * 55)
    for mol_name, smiles in molecules_dict.items():
        print(f'\n▶ {mol_name}')
        explain_ig(smiles, target_name=target_name, n_steps=n_steps)

gallery(MOLECULES, target_name='SR-p53')

In [ ]:
# Rerun for the nuclear estrogen receptor to compare structural patterns
gallery(MOLECULES, target_name='NR-ER')